# 2. Simulating data and training the networks

The TT (noise) network is trained first, then the AA (signal) network. This notebook uses a few hundred
simulations and few epochs so it runs in minutes; the paper used ~10⁵ simulations
(the same steps as `TT_channel/generate_data_and_train_TT_channel.py` and
`AA_channel/generate_data_and_train_AA_channel.py`). Simulations run in parallel at ~0.1 s each.

In [1]:
%matplotlib inline
import os, warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import torch
from sbi import utils
from torch.distributions.normal import Normal
from GWBackFinder import Simulator
from GWBackFinder import prior as GW_prior
from GWBackFinder import train as GW_train
from GWBackFinder.posterior import load_posterior, sample_two_step

sim = Simulator()
out = "training_demo"; os.makedirs(out, exist_ok=True)
n_simulations = 300
torch.manual_seed(0)

## TT network: noise amplitude P

Prior P ~ N(15, 3); the data are the log10 TT spectrum at the 2970 unbinned frequencies.

In [2]:
prior_TT, *_ = utils.process_prior(Normal(torch.tensor([15.0]), torch.tensor([0.2 * 15])))
thetas_TT = prior_TT.sample((n_simulations,))
x_TT = torch.tensor(sim.simulate_TT_batch(thetas_TT.numpy(), seed=0, n_jobs=os.cpu_count()), dtype=torch.float32)
print(thetas_TT.shape, x_TT.shape)

  0%|          | 0/300 [00:00<?, ?it/s]

 85%|████████▌ | 256/300 [00:00<00:00, 449.00it/s]

100%|██████████| 300/300 [00:00<00:00, 414.58it/s]

torch.Size([300, 1]) torch.Size([300, 2970])


In [3]:
GW_train.train(thetas=thetas_TT, gw_total=x_TT, prior=prior_TT, validation_fraction=0.2,
               learning_rate=1e-3, show_train_summary=False, max_num_epochs=30,
               path_inference=out, name_file="train_TT.pkl", model_type="nsf",
               hidden_features=64, num_transforms=3)
post_TT = GW_train.get_posterior(f"{out}/train_TT.pkl", f"{out}/posterior_TT.pkl")

## AA network: 26 slopes, amplitude and noise A

Parameters from `GWBackFinder.prior.get_prior()`; the data are the log10 AA spectrum (3970 points)
with the true P appended (P is taken from the TT network at inference time).

In [4]:
prior_AA = GW_prior.get_prior()
thetas_AA = prior_AA.sample((n_simulations,))
x_AA = torch.tensor(sim.simulate_AA_batch(thetas_AA.numpy(), seed=0, n_jobs=os.cpu_count()), dtype=torch.float32)
print(thetas_AA.shape, x_AA.shape)

  0%|          | 0/300 [00:00<?, ?it/s]

 85%|████████▌ | 256/300 [00:10<00:01, 24.16it/s]

100%|██████████| 300/300 [00:10<00:00, 28.19it/s]

torch.Size([300, 28]) torch.Size([300, 3971])


In [5]:
GW_train.train(thetas=thetas_AA, gw_total=x_AA, prior=prior_AA, validation_fraction=0.2,
               learning_rate=1e-3, show_train_summary=False, max_num_epochs=30,
               path_inference=out, name_file="train_AA.pkl", model_type="nsf",
               hidden_features=64, num_transforms=3)
post_AA = GW_train.get_posterior(f"{out}/train_AA.pkl", f"{out}/posterior_AA.pkl")

## Using the new networks

Exactly like the shipped ones (notebook 1). With this little training they are not accurate.

In [6]:
dAA, dTT, _, _ = sim.different_signals("powerlaw", [-10.0, 0.6], [15.0, 3.0], rng=0)
s, sP = sample_two_step(load_posterior(f"{out}/posterior_AA.pkl"), load_posterior(f"{out}/posterior_TT.pkl"),
                        dAA, dTT, n_samples=200)
print("P: median", np.median(sP).round(2), "(truth 15)")

P: median 14.99 (truth 15)


To continue training an existing inference object, see `TT_channel/resume_training_noise.py` and
`AA_channel/resume_training_signal.py` (`GW_train.train(..., resume_training=True, path_saved=...)`).

In [7]:
import shutil; shutil.rmtree(out)